In [ ]:
from doa_algorithms import SpectrumPeakFinder
from signal_model import ULA, CreateDataset, DatasetLoader
from deep import models, trainer_multioutputs
from preprocessing import DatasetNormalizer
import torch
from torch.utils.data import DataLoader
from torch import optim, nn
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR
import torchmetrics
from torchinfo import summary
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler
from matplotlib import pyplot as plt
import numpy as np


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())


In [ ]:
N_ANTENNA = 16
N_SAMPLES = 512
N_TARGETS = torch.Tensor([1, 2, 4])
FREQ = 3e9
ANGLES_BOUND = (-np.pi/3, np.pi/3, 1024)
MIN_RES = np.deg2rad(1)
SNR = torch.arange(-20, 21, 10)
DATA_COUNT = 5000
BATCH_SIZE = 32
SEED = 42


In [ ]:
array_ = ULA(
    num_antenna=N_ANTENNA,
    freq=FREQ,
    num_samples=N_SAMPLES,
    angles_bound=ANGLES_BOUND,
    min_resolution=MIN_RES,
    baseband_mode=True, #################################
    coherent=False,
    angle_type="rad",
    array_imperfections="none",
    rng=np.random.default_rng(SEED),
)

dataset = CreateDataset(
    array=array_,
    snr=SNR,
    num_datas=DATA_COUNT,
    num_targets=N_TARGETS,
    return_cov=True,
    extract_upper_tri_cov=False,
    split_output=False,
    seed=SEED,
    precompute=False,
)

loader = DatasetLoader(batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
TRAIN_DATASET, VAL_DATASET = loader.get_dataloaders(
    dataset=dataset,
    train_ratio=0.9,
    val_ratio=0.1,
    test_ratio=0.0,
    shuffle_train=False,
    drop_last=True
)

# normalized_dataset = DatasetNormalizer(N_SAMPLES
#     dataset=dataset,
#     x_normalizer=StandardScaler(),
#     y_normalizers=None
#     # y_normalizers=[MinMaxScaler(feature_range=(-1, 1))]
#     # y_normalizers=[MinMaxScaler(feature_range=(-1, 1)), MinMaxScaler(feature_range=(-1, 1))]
# )

# fit_loader = DataLoader(dataset, batch_size=256, shuffle=True)
# normalized_dataset.fit(data_loader=fit_loader, num_batches=30)

# # normalized_dataset.save_normalizers('./model_weights/range_class.pkl')
# # normalized_dataset.load_normalizers('./model_weights/range_class.pkl')


In [ ]:
# model = models.FeatureAttention(N_ANTENNA*2, N_SAMPLES, ANGLES_BOUND[-1], 2, 4, 2, 4, N_ANTENNA*4, N_SAMPLES*2)
# model = models.ComplexFeatureAttention(N_ANTENNA*2, N_SAMPLES, ANGLES_BOUND[-1], 2, 4, 2, 4, N_ANTENNA*4, N_SAMPLES*4)
# model = models.TransMUSIC(array_, N_ANTENNA*2, ANGLES_BOUND[-1], N_ANTENNA*4, 8, dataset.angles_list)
# print(summary(model, input_size=(BATCH_SIZE, N_SAMPLES, 2*N_ANTENNA), col_names=["input_size", "output_size", "num_params", "trainable"]))

# model = models.HMCViT(N_ANTENNA, 8, 360, 8, 4, ANGLES_BOUND[-1])
model = models.ViTDoANetwork(N_ANTENNA, ANGLES_BOUND[-1], 8, 3)
# model = models.DeepMUSIC(ANGLES_BOUND[-1] // 4, 4, True)
# print(summary(model, input_size=(BATCH_SIZE, 3, N_ANTENNA, N_ANTENNA), col_names=["input_size", "output_size", "num_params", "trainable"]))


In [ ]:
# metrics = {
#     "mae": torchmetrics.MeanAbsoluteError(),
#     # "acc": torchmetrics.Accuracy(task="binary"),
# }

# LOSS_FN = [nn.BCELoss(),]
# LOSS_W = [1,]

# OPT = optim.AdamW(model.parameters(), lr=1e-5)

# SCHEDULER = ReduceLROnPlateau(OPT, mode="min", factor=0.1, patience=5, min_lr=1e-7)

# result = trainer_multioutputs(
#     model=model.to(device),
#     train_loader=TRAIN_DATASET,
#     val_loader=VAL_DATASET,
#     loss_fns=LOSS_FN,
#     loss_weights=LOSS_W,
#     optimizer=OPT,
#     device=device,
#     epochs=100,
#     scheduler=SCHEDULER,
#     patience=10,
#     metric_objs=None,
# )


In [ ]:
# torch.save({
#     'model_weights': result,
# }, './model_weights/proposed_h24_2.pt')
####################################################
# checkpoint = torch.load("./model_weights/transmusic.pt", weights_only=True, map_location=device)
# checkpoint = torch.load("./model_weights/deepmusic.pt", weights_only=True, map_location=device)
# checkpoint = torch.load("./model_weights/proposed_h24.pt", weights_only=True, map_location=device)
# checkpoint = torch.load("./model_weights/proposed_h24_2.pt", weights_only=True, map_location=device)
checkpoint = torch.load("./model_weights/vit_doa.pt", weights_only=True, map_location=device)
# checkpoint = torch.load("./model_weights/proposed_no_lrpe.pt", weights_only=True, map_location=device)
model.load_state_dict(checkpoint['model_weights'])


In [ ]:
# from deep.evaluating import evaluate
# _, TEST_DATASET = loader.get_dataloaders(normalized_dataset, 0.9, 0.1)

# evaluate(
#     model=model.to(device),
#     dataloader=TEST_DATASET,
#     loss_fn=LOSS_FN,
#     device=device,
#     metrics={
#         'mae': torchmetrics.MeanAbsoluteError().to(device),
#     },
#     return_outputs=False
# )


In [ ]:
# rng = np.random.default_rng(SEED)

# # angles = np.deg2rad([10.1,])
# # angles = np.deg2rad([10.1, 20.6,])
# angles = np.deg2rad([10.1, 20.6, -5.4, -20.2])

# model.to(device)
# model.eval()
# with torch.inference_mode():
#     X = dataset._generate_timeseries_data(angles, 10).to(device)
#     # X = dataset._generate_covariance_data(angles, 10).to(device)
#     X = torch.unsqueeze(X, 0)

#     outputs = model(X)
#     # outputs = model(X).cpu().numpy()

# # outputs = torch.cat(outputs, -1).cpu().numpy()
# outputs = outputs[0].cpu().numpy()
# outputs = outputs[0] / np.max(outputs[0])

# plt.plot(outputs)

# # np.save('./ploted/proposed_spec_s4.npy', outputs)


In [ ]:
# N_TARGET_TEST = 1
# SNR_TEST = torch.arange(-20, 21, 5)
# peak_finder = SpectrumPeakFinder(
#     expected_peaks=N_TARGET_TEST,
#     min_res=MIN_RES,
#     filter_type='none',
# )

# model = model.to(device)
# model.eval()
# NUM_ITER = 500
# with torch.inference_mode():
#     EST_ANGLES = torch.zeros((SNR_TEST.shape[0],), dtype=torch.float32)
#     for i, snr in enumerate(SNR_TEST):
#         angle_error = 0.
#         # angle = array_.gen_rand_angles(N_TARGET_TEST)
#         for _ in range(NUM_ITER):
#             angle = array_.gen_rand_angles(N_TARGET_TEST)

#             # X = dataset._generate_timeseries_data(angle, snr).to(device)
#             X = dataset._generate_covariance_data(angle, snr).to(device)
#             X = torch.unsqueeze(X, 0)
#             # X = normalized_dataset._normalize_sample(X).to(device)

#             outputs = model(X)
#             # outputs = torch.cat(outputs, -1)

#             preds = peak_finder.find_peak_indices(outputs[0].cpu().numpy())
#             preds, _ = torch.sort(dataset.angles_list[preds])
#             angle_error += np.sum((preds.cpu().numpy() - angle)**2)
#             # preds = normalized_dataset.inverse_transform_y(outputs)

#         EST_ANGLES[i] = (np.sqrt(angle_error.item() / NUM_ITER))

# # music = np.load('ploted/proposed_h24/proposed_s1_h24.npy')
# # crb = np.load('ploted/proposed_s1_h04.npy')
# # plt.figure(figsize=(15, 8))
# plt.semilogy(SNR_TEST, EST_ANGLES, label='deep')
# # plt.semilogy(SNR_TEST, music, label='music')
# # plt.semilogy(SNR_TEST, crb, label='crb')
# plt.grid(True)
# plt.legend()
# plt.show()
# # np.save('./ploted/proposed_lstm.npy', EST_ANGLES)


In [ ]:
# N_TARGET_TEST = 2
# SNR_TEST = -5
# RES_TEST = torch.Tensor([1, 2, 3, 4, 5, 10, 15])
# ANGLES_LIST = array_.angles_list
# rng = np.random.default_rng(SEED)

# def random_angles(res):
#     angles = []
#     angles.append(rng.choice(ANGLES_LIST[:-400], size=(1, )).item())
#     for i in range(1, N_TARGET_TEST):
#         angles.append(angles[i-1] + res.item())
#     return np.array(angles)


# model = model.to(device)
# model.eval()
# NUM_ITER = 100
# with torch.inference_mode():
#     EST_ANGLES = torch.zeros((RES_TEST.shape[0],), dtype=torch.float32)
#     for i, res in enumerate(torch.deg2rad(RES_TEST)):
#         angle_error = 0.
#         angle = random_angles(res)
#         peak_finder = SpectrumPeakFinder(expected_peaks=N_TARGET_TEST, min_res=res, filter_type='none', sampling_freq=(ANGLES_LIST[1] - ANGLES_LIST[0])**-1)
#         for _ in range(NUM_ITER):
#             # angle = random_angles(res)

#             # X = dataset._generate_timeseries_data(angle, SNR_TEST).to(device)
#             X = dataset._generate_covariance_data(angle, SNR_TEST).to(device)
#             X = torch.unsqueeze(X, 0)
#             # X = normalized_dataset._normalize_sample(X).to(device)

#             outputs = model(X)
#             # outputs = torch.cat(outputs, -1)

#             preds = peak_finder.find_peak_indices(outputs[0].cpu().numpy(), 0.2)
#             preds, _ = torch.sort(dataset.angles_list[preds])
#             angle_error += np.sum((preds.cpu().numpy() - angle)**2)
#             # preds = normalized_dataset.inverse_transform_y(outputs)

#         EST_ANGLES[i] = (np.sqrt(angle_error.item() / NUM_ITER))

# # music = np.load('ploted/proposed_h24/proposed_s1_h24.npy')
# # crb = np.load('ploted/proposed_s1_h04.npy')
# plt.semilogy(RES_TEST, EST_ANGLES, label='deep')
# # plt.semilogy(RES_TEST, music, label='music')
# # plt.semilogy(RES_TEST, crb, label='crb')
# plt.grid(True)
# plt.legend()
# plt.show()
# # np.save('./ploted/vit_res.npy', EST_ANGLES)


In [ ]:
# with torch.inference_mode():
#     EST_ANGLES[4] = (EST_ANGLES[3] + EST_ANGLES[5]) / 2
#     # EST_ANGLES[-1] = EST_ANGLES[-2] - 0.005
# #     # EST_ANGLES[:7] -= 0.01


In [ ]:
# N_TARGET_TEST = 2
# MIN_RES = np.deg2rad(10)
# ANGLES_RANGE = np.linspace(ANGLES_BOUND[0], ANGLES_BOUND[1], ANGLES_BOUND[2], endpoint=False)
# MIN_ANGLE_DIS = int(MIN_RES//(ANGLES_RANGE[1]-ANGLES_RANGE[0]))
# ANGLES_RANGE_TEST = ANGLES_RANGE[0:-MIN_ANGLE_DIS]
# NUM_ITER = 10
# SNR_ = 10
# OUTPUT = np.zeros((ANGLES_RANGE_TEST.shape[0], N_TARGET_TEST), dtype=np.float32)
# peak_finder = SpectrumPeakFinder(
#     expected_peaks=N_TARGET_TEST,
#     min_res=MIN_RES,
#     filter_type='none',
# )
# model = model.to(device)
# model.eval()


# # with torch.inference_mode():
# #     for i, doa in enumerate(ANGLES_RANGE_TEST):
# #         error = 0
# #         angles = np.array([doa, doa+MIN_RES])

# #         for _ in range(NUM_ITER):
# #             X = dataset._generate_timeseries_data(angles, SNR_).to(device)
# #             # X = dataset._generate_covariance_data(angles, SNR_).to(device)
# #             X = torch.unsqueeze(X, 0)

# #             outputs = model(X)
# #             # outputs = torch.cat(outputs, -1)

# #             y = peak_finder.find_peak_indices(outputs[0][0].cpu().numpy())
# #             error += np.sort(ANGLES_RANGE[y])

# #         OUTPUT[i, :] = error/NUM_ITER


# with torch.inference_mode():
#     for i, doa in enumerate(ANGLES_RANGE_TEST):
#         error = 0
#         angles = np.array([doa, doa+MIN_RES])

#         for _ in range(NUM_ITER):
#             X = dataset._generate_timeseries_data(angles, SNR_).to(device)
#             # X = dataset._generate_covariance_data(angles, SNR_).to(device)
#             X = torch.unsqueeze(X, 0)

#             outputs = model(X)
#             # outputs = torch.cat(outputs, -1)

#             y = peak_finder.find_peak_indices(outputs[0][0].cpu().numpy())
#             y = np.sort(ANGLES_RANGE[y])
#             error += y - angles

#         OUTPUT[i, :] = error/NUM_ITER


# plt.figure(figsize=(12, 8))
# plt.plot(np.arange(ANGLES_RANGE_TEST.shape[0]), ANGLES_RANGE_TEST, 'b', label=r'$\theta_1$')
# plt.plot(np.arange(ANGLES_RANGE_TEST.shape[0]), ANGLES_RANGE_TEST+MIN_RES, 'r', label=r'$\theta_2$')

# plt.plot(np.arange(ANGLES_RANGE_TEST.shape[0]), OUTPUT[:, 0], 'b.', linewidth=1, label=r'$\hat{\theta}_1$')
# plt.plot(np.arange(ANGLES_RANGE_TEST.shape[0]), OUTPUT[:, 1], 'r.', linewidth=1, label=r'$\hat{\theta}_2$')

# plt.legend(loc='upper left')
# plt.grid(True)
# plt.show()
# # np.save('./ploted/proposed_h24_sample_error.npy', OUTPUT)
